## Mock practical assessment 01

Read `BRIEF.md` before starting. Select `.venv-py311`. This notebook belongs in `mock_assessment_01/`.

Load the supplied files yourself and work through the tasks. Record elapsed time at submission.

## 1. Inspection and cleaning

Show your inspection, cleaning decisions, row counts, lookup join and unmatched-key check.

In [1]:
import pandas as pd
plans_raw = pd.read_csv("data/plans.csv")
customers_raw = pd.read_csv("data/customer_snapshots.csv")

plans = plans_raw.copy()
customers = customers_raw.copy()

customers.info()
customers.isna().sum()
plans

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 734 entries, 0 to 733
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               734 non-null    object 
 1   snapshot_date             734 non-null    object 
 2   country                   717 non-null    object 
 3   plan_id                   734 non-null    object 
 4   contract                  702 non-null    object 
 5   tenure_months             695 non-null    float64
 6   monthly_charge            734 non-null    object 
 7   usage_hours_last_30d      690 non-null    float64
 8   support_tickets_last_30d  734 non-null    int64  
 9   churn_next_90d            726 non-null    float64
 10  cancellation_date         186 non-null    object 
dtypes: float64(3), int64(1), object(7)
memory usage: 63.2+ KB


,plan_id,plan_name,list_price
0,P1,Basic,29
1,P2,Plus,59
2,P3,Premium,99
3,P2,Plus,59


In [2]:
customers["monthly_charge"].nunique()

629

Let's identify which column needs unique value
- customer_id because one customer can only subscribe to one plan at a time

In [3]:
customers.dtypes


customer_id                  object
snapshot_date                object
country                      object
plan_id                      object
contract                     object
tenure_months               float64
monthly_charge               object
usage_hours_last_30d        float64
support_tickets_last_30d      int64
churn_next_90d              float64
cancellation_date            object
dtype: object

In [4]:
customers.isna().sum() 

customers.isna().mean() * 100

# customers[customers["usage_hours_last_30d"].isna()]


customer_id                  0.000000
snapshot_date                0.000000
country                      2.316076
plan_id                      0.000000
contract                     4.359673
tenure_months                5.313351
monthly_charge               0.000000
usage_hours_last_30d         5.994550
support_tickets_last_30d     0.000000
churn_next_90d               1.089918
cancellation_date           74.659401
dtype: float64

In [5]:
customers.shape

(734, 11)

Replace null values in these columns with
- Country: 'Unknown'
- Contract: 'Unknown'
- tenure_months: mean value for the country ; calculate as proportion of missing vals, and flag if % is high e.g. greater than 5%
- usage_hours_last_30d : mean value for each country

Drop na in target columnd (churn_next_90d)

In [6]:
customers.columns

Index(['customer_id', 'snapshot_date', 'country', 'plan_id', 'contract',
       'tenure_months', 'monthly_charge', 'usage_hours_last_30d',
       'support_tickets_last_30d', 'churn_next_90d', 'cancellation_date'],
      dtype='object')

In [7]:
customers["contract"] = customers["contract"].str.lower().str.strip().str.title()

In [8]:
customers["contract"].value_counts()

contract
Monthly    469
Annual     233
Name: count, dtype: int64

In [9]:
pd.crosstab(
    customers["cancellation_date"].isna(),
    customers["churn_next_90d"]
)

churn_next_90d,0.0,1.0
cancellation_date,,
False,0,185
True,541,0


Suspect that cancellation_date could occur after snapshot date, which could indicate target leakage. we will verify that in the following block.

In [10]:
help(customers.where)

Help on method where in module pandas.core.generic:

where(cond, other=nan, *, inplace: 'bool_t' = False, axis: 'Axis | None' = None, level: 'Level | None' = None) -> 'Self | None' method of pandas.core.frame.DataFrame instance
    Replace values where the condition is False.
    
    Parameters
    ----------
    cond : bool Series/DataFrame, array-like, or callable
        Where `cond` is True, keep the original value. Where
        False, replace with corresponding value from `other`.
        If `cond` is callable, it is computed on the Series/DataFrame and
        should return boolean Series/DataFrame or array. The callable must
        not change input Series/DataFrame (though pandas doesn't check it).
    other : scalar, Series/DataFrame, or callable
        Entries where `cond` is False are replaced with
        corresponding value from `other`.
        If other is callable, it is computed on the Series/DataFrame and
        should return scalar or Series/DataFrame. The callabl

In [11]:
customers[["cancellation_date", "snapshot_date"]].loc[customers["cancellation_date"].notna()]

,cancellation_date,snapshot_date
5,2026-09-01,2026-06-30
9,2026-09-09,2026-06-30
13,2026-08-02,2026-06-30
14,2026-08-31,2026-06-30
20,2026-08-08,2026-06-30
...,...,...
709,2026-09-26,2026-06-30
714,2026-08-03,2026-06-30
723,2026-09-08,2026-06-30
726,2026-09-24,2026-06-30


In [12]:
customers.columns

plans = plans.drop_duplicates()
plans


,plan_id,plan_name,list_price
0,P1,Basic,29
1,P2,Plus,59
2,P3,Premium,99


In [13]:
customers["snapshot_date"] = pd.to_datetime(customers["snapshot_date"])

customers["cancellation_date"] = pd.to_datetime(customers["cancellation_date"])

(
    customers.loc[customers["cancellation_date"].notna(), "cancellation_date"] >
    customers.loc[customers["cancellation_date"].notna(), "snapshot_date"]
).value_counts()

# customers["cancellation_date"].value_counts(dropna=False)


True    186
Name: count, dtype: int64

In [14]:
days_until_cancel = (
    customers["cancellation_date"] - customers["snapshot_date"]
).dt.days

days_until_cancel.describe()

count    186.000000
mean      44.763441
std       24.892590
min        1.000000
25%       24.250000
50%       45.000000
75%       64.000000
max       90.000000
dtype: float64

calculating country means on the entire dataset before splitting still causes data leakage

split the data first and learn imputation statistics exclusively from the training set. 

Use the same country_means for X_test

X_train determines teh country means
X_test only looks up those previosuly learned values

This prevents information from teh test dataset influencing your preprocessing

series.map(mapping)
- groupby().mean() followed by .map()

In [15]:
customers[["country", "contract"]] = customers[["country", "contract"]].fillna("Unknown")

# customers["tenure_months"] = customers.groupby("country")["tenure_months"].transform("mean")
# customers["tenure_months"] = customers["tenure_months"].fillna(
#     customers.groupby("country")["tenure_months"].transform("mean")
# )

# customers["usage_hours_last_30d"] = customers["usage_hours_last_30d"].fillna(
#     customers.groupby("country")["usage_hours_last_30d"].transform("mean")
# )

# shouldn't find the mean of the group before splitting, if customers 
# represents the training split already then it's fine

customers.isna().mean() * 100

#.mean() calculates the proportion of msising values

customer_id                  0.000000
snapshot_date                0.000000
country                      0.000000
plan_id                      0.000000
contract                     0.000000
tenure_months                5.313351
monthly_charge               0.000000
usage_hours_last_30d         5.994550
support_tickets_last_30d     0.000000
churn_next_90d               1.089918
cancellation_date           74.659401
dtype: float64

In [16]:
customers = customers.drop(columns=["cancellation_date"])

customers.columns

Index(['customer_id', 'snapshot_date', 'country', 'plan_id', 'contract',
       'tenure_months', 'monthly_charge', 'usage_hours_last_30d',
       'support_tickets_last_30d', 'churn_next_90d'],
      dtype='object')

In [17]:
customers["country"] = customers["country"].str.lower().str.strip().str.capitalize()

In [18]:
customers[customers["customer_id"] == "C0472"]

customers.groupby("customer_id")["plan_id"].nunique().value_counts()

plan_id
1    720
Name: count, dtype: int64

In [19]:
customers.groupby("plan_id", as_index=False)["customer_id"].nunique().value_counts()

plan_id  customer_id
P1       317            1
P2       249            1
P3       149            1
P9       5              1
Name: count, dtype: int64

In [20]:
customers_plan = customers.merge(
    plans, 
    on="plan_id",
    how="left",
    validate="m:1",
    indicator=True

)

In [21]:
unmatched = customers_plan[
    customers_plan["_merge"] == "left_only"
]

# mark their unmacttehd plan info as unknown

customers_plan.loc[
    customers_plan["_merge"] == "left_only",
    "plan_name"
] = "Unknown"

assert customers_plan["plan_name"].notna().all()

In [22]:
import numpy as np

customers_plan["list_price"] = (
    customers_plan["list_price"].replace("Unknown", np.nan)
)


In [23]:
original = customers_plan["list_price"].copy()

cleaned = original.astype("string").str.replace("$", "", regex=False)

parsed = pd.to_numeric(
    original, errors="coerce"
)

failed = parsed.isna() & original.notna()

customers_plan.loc[failed, ["list_price"]]
print(type(original))

original[failed].value_counts(dropna=False)

customers_plan["list_price"] = parsed



<class 'pandas.core.series.Series'>


In [24]:
# customers_plan["list_price"] = (
#     customers_plan["list_price"].mask(
#         customers_plan["list_price"] == "Unknown"
#     )
# )

In [25]:
# customers_plan["list_price"] = (
#     customers_plan["list_price"].where(
#         customers_plan["list_price"] != "Unknown"
#     )
# )

In [26]:
# customers_plan["list_price"] = pd.to_numeric(
#     customers_plan["list_price"],
#     errors="coerce"
# )

In [27]:
help(customers.mask)

Help on method mask in module pandas.core.generic:

mask(cond, other=<no_default>, *, inplace: 'bool_t' = False, axis: 'Axis | None' = None, level: 'Level | None' = None) -> 'Self | None' method of pandas.core.frame.DataFrame instance
    Replace values where the condition is True.
    
    Parameters
    ----------
    cond : bool Series/DataFrame, array-like, or callable
        Where `cond` is False, keep the original value. Where
        True, replace with corresponding value from `other`.
        If `cond` is callable, it is computed on the Series/DataFrame and
        should return boolean Series/DataFrame or array. The callable must
        not change input Series/DataFrame (though pandas doesn't check it).
    other : scalar, Series/DataFrame, or callable
        Entries where `cond` is True are replaced with
        corresponding value from `other`.
        If other is callable, it is computed on the Series/DataFrame and
        should return scalar or Series/DataFrame. The ca

In [28]:
customers_plan["list_price"] = pd.to_numeric(
    customers_plan["list_price"], 
    errors="coerce"
)

# errors="coerce" converts unexpected invalid values to NaN

In [29]:
customers_plan.isna().sum()

customer_id                  0
snapshot_date                0
country                      0
plan_id                      0
contract                     0
tenure_months               39
monthly_charge               0
usage_hours_last_30d        44
support_tickets_last_30d     0
churn_next_90d               8
plan_name                    0
list_price                   6
_merge                       0
dtype: int64

In [30]:
customers_plan[customers_plan["list_price"].isna()]

,customer_id,snapshot_date,country,plan_id,contract,tenure_months,monthly_charge,usage_hours_last_30d,support_tickets_last_30d,churn_next_90d,plan_name,list_price,_merge
284,C0191,2026-06-30,Singapore,P9,Annual,23.0,69.71,5.9,1,0.0,Unknown,NaN,left_only
500,C0173,2026-06-30,Thailand,P9,Monthly,33.0,30.9,9.5,2,0.0,Unknown,NaN,left_only
507,C0416,2026-06-30,Singapore,P9,Unknown,20.0,26.78,21.6,0,0.0,Unknown,NaN,left_only
543,C0416,2026-06-30,Singapore,P9,Unknown,20.0,26.78,21.6,0,0.0,Unknown,NaN,left_only
696,C0009,2026-06-30,Thailand,P9,Annual,5.0,104.81,17.2,0,0.0,Unknown,NaN,left_only
701,C0569,2026-06-30,Malaysia,P9,Monthly,65.0,25.98,18.8,0,0.0,Unknown,NaN,left_only


In [31]:
assert len(customers_plan) == len(customers)

In [32]:
assert customers_plan["customer_id"].notna().all()

In [33]:
customers.describe()

,snapshot_date,tenure_months,usage_hours_last_30d,support_tickets_last_30d,churn_next_90d
count,734,695.000000,690.000000,734.000000,726.000000
mean,2026-06-30 00:00:00,37.231655,14.646232,1.197548,0.254821
min,2026-06-30 00:00:00,1.000000,0.000000,0.000000,0.000000
25%,2026-06-30 00:00:00,20.000000,9.600000,0.000000,0.000000
50%,2026-06-30 00:00:00,38.000000,14.150000,1.000000,0.000000
75%,2026-06-30 00:00:00,55.000000,19.400000,2.000000,1.000000
max,2026-06-30 00:00:00,72.000000,34.000000,6.000000,1.000000
std,NaN,20.665711,7.026681,1.131441,0.436061


In [34]:
customers_plan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 734 entries, 0 to 733
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               734 non-null    object        
 1   snapshot_date             734 non-null    datetime64[ns]
 2   country                   734 non-null    object        
 3   plan_id                   734 non-null    object        
 4   contract                  734 non-null    object        
 5   tenure_months             695 non-null    float64       
 6   monthly_charge            734 non-null    object        
 7   usage_hours_last_30d      690 non-null    float64       
 8   support_tickets_last_30d  734 non-null    int64         
 9   churn_next_90d            726 non-null    float64       
 10  plan_name                 734 non-null    object        
 11  list_price                728 non-null    float64       
 12  _merge                

In [35]:
customers["churn_next_90d"].value_counts(dropna=False)

churn_next_90d
0.0    541
1.0    185
NaN      8
Name: count, dtype: int64

- remove duplicated row in plans data

In [36]:
customers_plan = customers_plan.drop_duplicates(subset=["customer_id"])

In [37]:
assert customers_plan["customer_id"].is_unique

In [38]:
customers_plan[customers_plan["customer_id"].duplicated(keep=False)].sort_values(by="customer_id")

,customer_id,snapshot_date,country,plan_id,contract,tenure_months,monthly_charge,usage_hours_last_30d,support_tickets_last_30d,churn_next_90d,plan_name,list_price,_merge


## 2. Analysis and SQL

Calculate the requested country summary in SQL and interpret its denominators, coverage and limitations.

In [39]:
customers["country"] = customers["country"].str.lower().str.strip().str.capitalize()

In [40]:
# preserve original values
original = customers_plan["monthly_charge"].copy()

# remove $ and whitespace
cleaned = (
    original.astype("string")
    .str.replace("$", "", regex=False)
# need regex=FAlse to treat $ as a literal character because
# $ has a special meaning' the end of string
)

#2. convert to numeric
parsed = pd.to_numeric(cleaned, errors="coerce")

# 3 Identify conversion failures
failed = original.notna() & parsed.isna()

invalid_amounts = parsed.notna() & (parsed <= 0)

# 4. Inspect failed rows
customers_plan.loc[failed, ["monthly_charge"]]

# customers_plan.loc[invalid_amount, [""]]
# count conversion failures

print("conversion failures:", failed.sum())

customers_plan.loc[invalid_amounts]

# convert invalid amounts to NaN
parsed = parsed.mask(invalid_amounts)

#.mask() replaces values with NaN wherever the condition is True

# 8. update DataFrame

customers_plan["monthly_charge"]= parsed

conversion failures: 13


In [41]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(":memory:")

customers.to_sql(name="customers", con=conn, index=False)
plans.to_sql(name="plans", con=conn, index=False)
customers_plan.to_sql("customers_plan", conn, index=False)

pd.read_sql("select * from customers_plan", conn).head()

# help(customers.to_sql)

,customer_id,snapshot_date,country,plan_id,contract,tenure_months,monthly_charge,usage_hours_last_30d,support_tickets_last_30d,churn_next_90d,plan_name,list_price,_merge
0,C0472,2026-06-30 00:00:00,Thailand,P2,Monthly,42.0,58.16,26.6,0,0.0,Plus,59.0,both
1,C0217,2026-06-30 00:00:00,Malaysia,P2,Monthly,26.0,60.28,12.6,1,0.0,Plus,59.0,both
2,C0285,2026-06-30 00:00:00,Malaysia,P1,Annual,68.0,29.53,17.5,1,0.0,Basic,29.0,both
3,C0381,2026-06-30 00:00:00,Thailand,P1,Monthly,39.0,NaN,10.5,0,0.0,Basic,29.0,both
4,C0295,2026-06-30 00:00:00,Thailand,P1,Monthly,27.0,26.99,0.0,0,0.0,Basic,29.0,both


In [42]:
(customers_plan["monthly_charge"] == "unknown").sum()

np.int64(0)

customers

In [43]:
query = """
    select coalesce(country, "Unknown") as country, count(customer_id) as total_customers, coalesce(sum(monthly_charge), 0) as total_monthly_charge
    , count(monthly_charge) as monthly_charge_coverage, avg(monthly_charge) as average_monthly_charge, count(churn_next_90d) as known_outcomes, sum(churn_next_90d) as churn_count, avg(churn_next_90d) * 100 as churn_rate_per_country 
      from customers_plan group by coalesce(country, "Unknown") order by total_monthly_charge desc
"""

pd.read_sql(query, conn)

,country,total_customers,total_monthly_charge,monthly_charge_coverage,average_monthly_charge,known_outcomes,churn_count,churn_rate_per_country
0,Thailand,347,18011.80,337,53.447478,340,90.0,26.470588
1,Malaysia,185,10071.28,181,55.642431,184,45.0,24.456522
2,Singapore,171,8672.38,165,52.559879,171,40.0,23.391813
3,Unknown,17,912.50,17,53.676471,17,7.0,41.176471


One pattern with sample sizes
One 

Inisghts
- on avereage, churn rate is between 22 and 26 % across Thailand, Malaysia, and Sigapore
- on avereage, monthly chage is around 53 
- 
- Thailand leads with the higest tota

Write your reasoning here.

## 3. Baseline and model

Define features and target, split, construct preprocessing, fit the baseline and model, and evaluate.

In [44]:
from sklearn.model_selection import train_test_split

# exclude identifiers

customers_plan = customers_plan.dropna(subset=["churn_next_90d"])

X = customers_plan.drop(columns=["churn_next_90d", "customer_id", "snapshot_date"])
y = customers_plan["churn_next_90d"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)

customers_plan.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 733
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               712 non-null    object        
 1   snapshot_date             712 non-null    datetime64[ns]
 2   country                   712 non-null    object        
 3   plan_id                   712 non-null    object        
 4   contract                  712 non-null    object        
 5   tenure_months             675 non-null    float64       
 6   monthly_charge            692 non-null    Float64       
 7   usage_hours_last_30d      668 non-null    float64       
 8   support_tickets_last_30d  712 non-null    int64         
 9   churn_next_90d            712 non-null    float64       
 10  plan_name                 712 non-null    object        
 11  list_price                707 non-null    float64       
 12  _merge                    7

In [45]:
assert "churn_next_90d" not in X.columns
assert "cancellation_date" not in X.columns

assert y.notna().all()
assert set(y.unique()).issubset({0, 1})
assert y.isin([0, 1]).all()
assert pd.Series(y.unique()).isin([0, 1]).all()

# assert expects a single boolean value, not an entire Series
# adding .all() converts the Boolean Series into one result

In [46]:
help(customers_plan.select_dtypes)

Help on method select_dtypes in module pandas.core.frame:

select_dtypes(include=None, exclude=None) -> 'Self' method of pandas.core.frame.DataFrame instance
    Return a subset of the DataFrame's columns based on the column dtypes.
    
    Parameters
    ----------
    include, exclude : scalar or list-like
        A selection of dtypes or strings to be included/excluded. At least
        one of these parameters must be supplied.
    
    Returns
    -------
    DataFrame
        The subset of the frame including the dtypes in ``include`` and
        excluding the dtypes in ``exclude``.
    
    Raises
    ------
    ValueError
        * If both of ``include`` and ``exclude`` are empty
        * If ``include`` and ``exclude`` have overlapping elements
        * If any kind of string dtype is passed in.
    
    See Also
    --------
    DataFrame.dtypes: Return Series with the data type of each column.
    
    Notes
    -----
    * To select all *numeric* types, use ``np.number`` or

In [47]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression


numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

In [48]:
X[numerical_features]

,tenure_months,monthly_charge,usage_hours_last_30d,support_tickets_last_30d,list_price
0,42.0,58.16,26.6,0,59.0
1,26.0,60.28,12.6,1,59.0
2,68.0,29.53,17.5,1,29.0
3,39.0,<NA>,10.5,0,29.0
4,27.0,26.99,0.0,0,29.0
...,...,...,...,...,...
728,69.0,59.71,NaN,0,59.0
729,19.0,24.99,18.2,0,29.0
730,8.0,59.76,9.9,1,59.0
731,12.0,61.15,11.6,0,59.0


In [49]:
X[categorical_features]

,country,plan_id,contract,plan_name,_merge
0,Thailand,P2,Monthly,Plus,both
1,Malaysia,P2,Monthly,Plus,both
2,Malaysia,P1,Annual,Basic,both
3,Thailand,P1,Monthly,Basic,both
4,Thailand,P1,Monthly,Basic,both
...,...,...,...,...,...
728,Singapore,P2,Monthly,Plus,both
729,Malaysia,P1,Monthly,Basic,both
730,Singapore,P2,Monthly,Plus,both
731,Malaysia,P2,Annual,Plus,both


In [50]:
numerical_features

['tenure_months',
 'monthly_charge',
 'usage_hours_last_30d',
 'support_tickets_last_30d',
 'list_price']

In [51]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, accuracy_score, classification_report

numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

dummy_model = Pipeline([
    # ("preprocessor", preprocessor), preprocessing for dummyclassifier is unnecessary because it ignores features
    # values and predicts only teh majority class learned from y_train
    ("classifier", DummyClassifier(strategy="most_frequent"))
])
dummy_model.fit(X_train, y_train)
dummy_model_predictions = dummy_model.predict(X_test)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])
model.fit(X_train, y_train)
model_predictions = model.predict(X_test)

print("model confusion matrix:\n", confusion_matrix(y_test, model_predictions))
print("model accuracy: ", accuracy_score(y_test, model_predictions))
print("dummy model confusion matrix:\n", confusion_matrix(y_test, dummy_model_predictions))
print("dummy model accuracy: ", accuracy_score(y_test, dummy_model_predictions))




model confusion matrix:
 [[104   2]
 [ 30   7]]
model accuracy:  0.7762237762237763
dummy model confusion matrix:
 [[106   0]
 [ 37   0]]
dummy model accuracy:  0.7412587412587412


Our model correctly classified 104 customers as non-churners and 7 as churners. However, it incorrectly flagged 2 customers as likely to churn and failed to identify 30 customers who actually churned. 

dummy model correctly classified 106 customers as non-churners but failed to identify any of the 37 customers who actually churned

Regression model performed slightly better with 74.8% accuracy compared to the dummy model's 74.1%

<!-- Despite making different classification errors, both models achieved the same accuracy of 74.31%. This is because they correctly classified the same numebr of customers, even though their predictions differed  -->

In [52]:
print("model classification report:\n", classification_report(y_test, model_predictions, zero_division=0))
print("dummy model classification report:\n", classification_report(y_test, dummy_model_predictions, zero_division=0))


model classification report:
               precision    recall  f1-score   support

         0.0       0.78      0.98      0.87       106
         1.0       0.78      0.19      0.30        37

    accuracy                           0.78       143
   macro avg       0.78      0.59      0.59       143
weighted avg       0.78      0.78      0.72       143

dummy model classification report:
               precision    recall  f1-score   support

         0.0       0.74      1.00      0.85       106
         1.0       0.00      0.00      0.00        37

    accuracy                           0.74       143
   macro avg       0.37      0.50      0.43       143
weighted avg       0.55      0.74      0.63       143



model achieved a 78% precision and 98% recall for non-churners, compared with 78% precision and 11=9% recall for churners. 

In contrast, dummy model achieved 74% precision and 100% recall for non-churners but 0% recall for churners, as it predicted every customer would stay.

Logistic regression also achieved a higher macro F1-score (0.59 versus 0.43), indicating better performance when both classes are weighted equally. 

Despite performing well at identifying non-churners and churners, approximately 81% of actual churners were missed. This limits the model's usefulness for retention targeting, where identifying at-risk customers is a key objective

Accuracy alone is insufficient because it measures overall correctness without distinguishing between different types of errors (fp and fn) or accounting for their business importance
- doesn't distinguish fp from fn: treats both mistakes equally, even when there business consequences differ
- doesn't account for class imbalance: can be dominated by the majority class (has more observations)


In our dataset, approximately 74% of customers did not churn, allowing the dummy model to achieve 74% accuracy simply by predicting that every custoemr would stay. However, this model failed to identify any actual churners. Since our business objective is to identify customers at risk of churning while minimising unnecessary contacts, precision and recall provide more relevant measures of model performane

In [53]:
import sklearn
# help(sklearn.metrics)
# classification_report?
help(DummyClassifier)

Help on class DummyClassifier in module sklearn.dummy:

class DummyClassifier(sklearn.base.MultiOutputMixin, sklearn.base.ClassifierMixin, sklearn.base.BaseEstimator)
 |  DummyClassifier(*, strategy='prior', random_state=None, constant=None)
 |  
 |  DummyClassifier makes predictions that ignore the input features.
 |  
 |  This classifier serves as a simple baseline to compare against other more
 |  complex classifiers.
 |  
 |  The specific behavior of the baseline is selected with the `strategy`
 |  parameter.
 |  
 |  All strategies make predictions that ignore the input feature values passed
 |  as the `X` argument to `fit` and `predict`. The predictions, however,
 |  typically depend on values observed in the `y` parameter passed to `fit`.
 |  
 |  Note that the "stratified" and "uniform" strategies lead to
 |  non-deterministic predictions that can be rendered deterministic by setting
 |  the `random_state` parameter if needed. The other strategies are naturally
 |  deterministi

In [54]:
help(ColumnTransformer)

Help on class ColumnTransformer in module sklearn.compose._column_transformer:

class ColumnTransformer(sklearn.base.TransformerMixin, sklearn.utils.metaestimators._BaseComposition)
 |  ColumnTransformer(transformers, *, remainder='drop', sparse_threshold=0.3, n_jobs=None, transformer_weights=None, verbose=False, verbose_feature_names_out=True, force_int_remainder_cols=True)
 |  
 |  Applies transformers to columns of an array or pandas DataFrame.
 |  
 |  This estimator allows different columns or column subsets of the input
 |  to be transformed separately and the features generated by each transformer
 |  will be concatenated to form a single feature space.
 |  This is useful for heterogeneous or columnar data, to combine several
 |  feature extraction mechanisms or transformations into a single transformer.
 |  
 |  Read more in the :ref:`User Guide <column_transformer>`.
 |  
 |  .. versionadded:: 0.20
 |  
 |  Parameters
 |  ----------
 |  transformers : list of tuples
 |      Li

In [55]:
OneHotEncoder?

Init signature:
OneHotEncoder(
    *,
    categories='auto',
    drop=None,
    sparse_output=True,
    dtype=<class 'numpy.float64'>,
    handle_unknown='error',
    min_frequency=None,
    max_categories=None,
    feature_name_combiner='concat',
)
Docstring:     
Encode categorical features as a one-hot numeric array.

The input to this transformer should be an array-like of integers or
strings, denoting the values taken on by categorical (discrete) features.
The features are encoded using a one-hot (aka 'one-of-K' or 'dummy')
encoding scheme. This creates a binary column for each category and
returns a sparse matrix or dense array (depending on the ``sparse_output``
parameter).

By default, the encoder derives the categories based on the unique values
in each feature. Alternatively, you can also specify the `categories`
manually.

This encoding is needed for feeding categorical data to many scikit-learn
estimators, notably linear models and SVMs with the standard kernels.

Note: a o

In [56]:
categorical_features

['country', 'plan_id', 'contract', 'plan_name', '_merge']

In [57]:
SimpleImputer?

Init signature:
SimpleImputer(
    *,
    missing_values=nan,
    strategy='mean',
    fill_value=None,
    copy=True,
    add_indicator=False,
    keep_empty_features=False,
)
Docstring:     
Univariate imputer for completing missing values with simple strategies.

Replace missing values using a descriptive statistic (e.g. mean, median, or
most frequent) along each column, or using a constant value.

Read more in the :ref:`User Guide <impute>`.

.. versionadded:: 0.20
   `SimpleImputer` replaces the previous `sklearn.preprocessing.Imputer`
   estimator which is now removed.

Parameters
----------
missing_values : int, float, str, np.nan, None or pandas.NA, default=np.nan
    The placeholder for the missing values. All occurrences of
    `missing_values` will be imputed. For pandas' dataframes with
    nullable integer dtypes with missing values, `missing_values`
    can be set to either `np.nan` or `pd.NA`.

strategy : str or Callable, default='mean'
    The imputation strategy.

    

In [58]:
import sklearn
help(sklearn.pipeline)
sklearn.pipeline?

Help on module sklearn.pipeline in sklearn:

NAME
    sklearn.pipeline - Utilities to build a composite estimator as a chain of transforms and estimators.

CLASSES
    sklearn.base.TransformerMixin(sklearn.utils._set_output._SetOutputMixin)
        FeatureUnion(sklearn.base.TransformerMixin, sklearn.utils.metaestimators._BaseComposition)
    sklearn.utils.metaestimators._BaseComposition(sklearn.base.BaseEstimator)
        FeatureUnion(sklearn.base.TransformerMixin, sklearn.utils.metaestimators._BaseComposition)
        Pipeline
    
    class FeatureUnion(sklearn.base.TransformerMixin, sklearn.utils.metaestimators._BaseComposition)
     |  FeatureUnion(transformer_list, *, n_jobs=None, transformer_weights=None, verbose=False, verbose_feature_names_out=True)
     |  
     |  Concatenates results of multiple transformer objects.
     |  
     |  This estimator applies a list of transformer objects in parallel to the
     |  input data, then concatenates the results. This is useful to com

Type:        module
String form: <module 'sklearn.pipeline' from 'c:\\Users\\ROG Zephyrus G14\\Downloads\\my_cool_projects\\datascience_practice\\.venv-py311\\Lib\\site-packages\\sklearn\\pipeline.py'>
File:        ~\downloads\my_cool_projects\datascience_practice\.venv-py311\lib\site-packages\sklearn\pipeline.py
Docstring:   Utilities to build a composite estimator as a chain of transforms and estimators.

In [59]:
customers_plan.head()

,customer_id,snapshot_date,country,plan_id,contract,tenure_months,monthly_charge,usage_hours_last_30d,support_tickets_last_30d,churn_next_90d,plan_name,list_price,_merge
0,C0472,2026-06-30,Thailand,P2,Monthly,42.0,58.16,26.6,0,0.0,Plus,59.0,both
1,C0217,2026-06-30,Malaysia,P2,Monthly,26.0,60.28,12.6,1,0.0,Plus,59.0,both
2,C0285,2026-06-30,Malaysia,P1,Annual,68.0,29.53,17.5,1,0.0,Basic,29.0,both
3,C0381,2026-06-30,Thailand,P1,Monthly,39.0,<NA>,10.5,0,0.0,Basic,29.0,both
4,C0295,2026-06-30,Thailand,P1,Monthly,27.0,26.99,0.0,0,0.0,Basic,29.0,both


Write your reasoning here.

## 4. Validation and conclusions

Include meaningful checks and your 4?6 sentence conclusion with two next actions.

choose checks tied to work you actually performed

<!-- My decision
- keep one record per customer
- left-join the plan lookup -->

Findings
- high precision and recall for non-churners, but moderate precision for churners with low recall

limitation
- Too few data samples with only 143 (106 + 37) in total
- class imbalances
- The results come from a single test split on synthetic data, and performance on future real customers remains uncertain

recommendation
- gather more data to train the model to improve its performance
- use k-folds cross validation to increase teh gernalisability of the data
 
- run a randomised pi;pt among eligible customers, assigning some to receive the retention offer and others to a control group, then compare retention and icnremental profit

Things i would investigate before launching a campaign is the plan_id p9, and 
- missing-data patterns, errors across customer segments
- contact costs
- campaign capacity

predictiing churn risk along dodes not establish whetehr the offer will prevent churn

Write your reasoning here.

## Submission notes

Elapsed time: 4 hrs

Resources used: asking chatgpt occassionally for syntaxes and thoughts

Unfinished work or difficulties